In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import torch
import pandas as pd
from src.config import Config
from src.utils.seed import set_seed
from src.utils.io_helpers import load_checkpoint, load_json, save_metrics_table
from src.dataset import get_dataloaders
from src.models.vanilla_ae import VanillaAutoencoder
from src.models.sparse_ae import SparseAutoencoder
from src.models.vae import VariationalAutoencoder
from src.models.vanilla_gan import VanillaGenerator
from src.models.dcgan import DCGANGenerator
from src.models.ddpm import SmallUNet, DDPM
from src.evaluation.metrics import compute_fid_score

set_seed(Config.SEED)
Config.ensure_dirs()
_, _, test_loader = get_dataloaders(
    batch_size=Config.BATCH_SIZE,
    seed=Config.SEED,
    num_workers=2 if torch.cuda.is_available() else 0,
)

# Instantiate & load trained checkpoints
vanilla_ae = VanillaAutoencoder(Config.INPUT_DIM, 256, Config.AE_LATENT_DIM)
load_checkpoint(vanilla_ae, Config.CHECKPOINT_DIR / "vanilla_ae.pth", Config.DEVICE)

sparse_ae = SparseAutoencoder(Config.INPUT_DIM, Config.SPARSE_AE_HIDDEN_DIM, Config.SPARSE_AE_LATENT_DIM)
load_checkpoint(sparse_ae, Config.CHECKPOINT_DIR / "sparse_ae.pth", Config.DEVICE)

vae = VariationalAutoencoder(Config.INPUT_DIM, 256, Config.VAE_LATENT_DIM)
load_checkpoint(vae, Config.CHECKPOINT_DIR / "vae.pth", Config.DEVICE)

vanilla_gan_gen = VanillaGenerator(Config.VANILLA_GAN_NOISE_DIM, Config.INPUT_DIM)
load_checkpoint(vanilla_gan_gen, Config.CHECKPOINT_DIR / "vanilla_gan_gen.pth", Config.DEVICE)

dcgan_gen = DCGANGenerator(Config.DCGAN_NOISE_DIM, Config.DCGAN_FEATURE_MAPS, Config.IMG_CHANNELS)
load_checkpoint(dcgan_gen, Config.CHECKPOINT_DIR / "dcgan_gen.pth", Config.DEVICE)

ddpm_unet = SmallUNet(Config.IMG_CHANNELS, Config.DDPM_BASE_CHANNELS, 128)
ddpm = DDPM(ddpm_unet, Config.DDPM_TIMESTEPS, Config.DDPM_BETA_START, Config.DDPM_BETA_END)
load_checkpoint(ddpm, Config.CHECKPOINT_DIR / "ddpm.pth", Config.DEVICE)

# Compute FID on 1,024 synthetic samples vs 7,000 Test Set images
NUM_FID_SAMPLES = 1024
print("Computing FID Scores on Test Set...")
fid_vae = compute_fid_score(test_loader, lambda bs: vae.sample(bs, Config.DEVICE), Config.DEVICE, NUM_FID_SAMPLES)
print(f"  VAE FID         : {fid_vae}")

fid_vgan = compute_fid_score(test_loader, lambda bs: vanilla_gan_gen.sample(bs, Config.DEVICE), Config.DEVICE, NUM_FID_SAMPLES)
print(f"  Vanilla GAN FID : {fid_vgan}")

fid_dcgan = compute_fid_score(test_loader, lambda bs: dcgan_gen.sample(bs, Config.DEVICE), Config.DEVICE, NUM_FID_SAMPLES)
print(f"  DCGAN FID       : {fid_dcgan}")

fid_ddpm = compute_fid_score(test_loader, lambda bs: ddpm.sample(bs, Config.DEVICE), Config.DEVICE, NUM_FID_SAMPLES, batch_size=256)
print(f"  DDPM FID        : {fid_ddpm}")

In [ ]:
res_vae_ae = load_json(Config.METRICS_DIR / "vanilla_ae_results.json")
res_sparse_ae = load_json(Config.METRICS_DIR / "sparse_ae_results.json")
res_vae = load_json(Config.METRICS_DIR / "vae_results.json")
res_vgan = load_json(Config.METRICS_DIR / "vanilla_gan_results.json")
res_dcgan = load_json(Config.METRICS_DIR / "dcgan_results.json")
res_ddpm = load_json(Config.METRICS_DIR / "ddpm_results.json")
res_mlp = load_json(Config.METRICS_DIR / "mlp_limitations_metrics.json")

records = [
    {
        "Arsitektur Model": "Vanilla Autoencoder",
        "Dim Laten": res_vae_ae["latent_dim"],
        "Training Time / Epoch (s)": res_vae_ae["avg_epoch_time_sec"],
        "Latency (ms / batch 64)": res_vae_ae["inference_latency_ms_batch64"],
        "Test MSE": res_vae_ae["test_metrics"]["mse"],
        "Test PSNR (dB)": res_vae_ae["test_metrics"]["psnr"],
        "Test SSIM": res_vae_ae["test_metrics"]["ssim"],
        "FID Score": "N/A (Latent Voids)",
        "Kestabilan Training": "Stabil / Konvergen",
    },
    {
        "Arsitektur Model": "Sparse Autoencoder",
        "Dim Laten": f"{res_sparse_ae['latent_dim']} (L1={res_sparse_ae['l1_lambda']})",
        "Training Time / Epoch (s)": res_sparse_ae["avg_epoch_time_sec"],
        "Latency (ms / batch 64)": res_sparse_ae["inference_latency_ms_batch64"],
        "Test MSE": res_sparse_ae["test_metrics"]["mse"],
        "Test PSNR (dB)": res_sparse_ae["test_metrics"]["psnr"],
        "Test SSIM": res_sparse_ae["test_metrics"]["ssim"],
        "FID Score": "N/A (Latent Voids)",
        "Kestabilan Training": "Stabil / Konvergen",
    },
    {
        "Arsitektur Model": "Variational Autoencoder (VAE)",
        "Dim Laten": res_vae["latent_dim"],
        "Training Time / Epoch (s)": res_vae["avg_epoch_time_sec"],
        "Latency (ms / batch 64)": res_vae["inference_latency_ms_batch64"],
        "Test MSE": res_vae["test_metrics"]["mse"],
        "Test PSNR (dB)": res_vae["test_metrics"]["psnr"],
        "Test SSIM": res_vae["test_metrics"]["ssim"],
        "FID Score": fid_vae,
        "Kestabilan Training": "Stabil / Konvergen",
    },
    {
        "Arsitektur Model": "Vanilla GAN (MLP)",
        "Dim Laten": res_vgan["latent_dim"],
        "Training Time / Epoch (s)": res_vgan["avg_epoch_time_sec"],
        "Latency (ms / batch 64)": res_vgan["inference_latency_ms_batch64"],
        "Test MSE": "N/A (Tanpa Encoder)",
        "Test PSNR (dB)": "N/A",
        "Test SSIM": "N/A",
        "FID Score": fid_vgan,
        "Kestabilan Training": "Rentan Osilasi / Mode Collapse",
    },
    {
        "Arsitektur Model": "Deep Convolutional GAN (DCGAN)",
        "Dim Laten": res_dcgan["latent_dim"],
        "Training Time / Epoch (s)": res_dcgan["avg_epoch_time_sec"],
        "Latency (ms / batch 64)": res_dcgan["inference_latency_ms_batch64"],
        "Test MSE": "N/A (Tanpa Encoder)",
        "Test PSNR (dB)": "N/A",
        "Test SSIM": "N/A",
        "FID Score": fid_dcgan,
        "Kestabilan Training": "Cukup Stabil (Sensitif Arsitektur)",
    },
    {
        "Arsitektur Model": "DDPM (U-Net, T=1000)",
        "Dim Laten": "28x28 (Dimensi Citra)",
        "Training Time / Epoch (s)": res_ddpm["avg_epoch_time_sec"],
        "Latency (ms / batch 64)": res_ddpm["inference_latency_ms_batch64"],
        "Test MSE": "N/A (Denoising Model)",
        "Test PSNR (dB)": "N/A",
        "Test SSIM": "N/A",
        "FID Score": fid_ddpm,
        "Kestabilan Training": "Sangat Stabil",
    },
]

df_table = save_metrics_table(records, Config.METRICS_DIR / "final_comparative_table.csv")
print("\nTABEL KOMPARASI METRIK NUMERIK (BAGIAN IV LAPORAN)")
display(df_table)

print("\nREFERENSI MLP FROZEN DECODER (BAGIAN III LAPORAN)")
print(f"MLP Classifier Test Accuracy: {res_mlp['classifier_test_acc']}%")
print(f"Frozen MLP Decoder Metrics  : {res_mlp['frozen_decoder_metrics']}")

In [ ]:
from src.evaluation.visualizations import (
    generate_figure_1_reconstructions,
    generate_figure_2_random_sampling,
    generate_figure_3_mlp_limitations,
    generate_figure_4_latent_interpolation,
    generate_figure_5_loss_and_ddpm_trajectory,
)

mlp_visuals = torch.load(Config.METRICS_DIR / "mlp_limitations_visuals.pt")
ddpm_snapshots = torch.load(Config.METRICS_DIR / "ddpm_trajectory_snapshots.pt")

histories = {
    "vanilla_ae": res_vae_ae["history"],
    "sparse_ae": res_sparse_ae["history"],
    "vae": res_vae["history"],
    "vanilla_gan": res_vgan["history"],
    "dcgan": res_dcgan["history"],
    "ddpm": res_ddpm["history"],
}

generate_figure_1_reconstructions(
    vanilla_ae, sparse_ae, vae, test_loader, Config.CLASS_NAMES, Config.DEVICE,
    Config.FIGURES_DIR / "fig1_reconstruction_grid.png",
)
generate_figure_2_random_sampling(
    vae, vanilla_gan_gen, dcgan_gen, ddpm, Config.DEVICE,
    Config.FIGURES_DIR / "fig2_random_sampling_grid.png",
)
generate_figure_3_mlp_limitations(
    mlp_visuals, vanilla_ae, Config.CLASS_NAMES, Config.DEVICE,
    Config.FIGURES_DIR / "fig3_mlp_limitations.png",
)
generate_figure_4_latent_interpolation(
    vanilla_ae, sparse_ae, vae, vanilla_gan_gen, dcgan_gen, test_loader, Config.DEVICE,
    Config.FIGURES_DIR / "fig4_latent_interpolation.png",
)
generate_figure_5_loss_and_ddpm_trajectory(
    histories, ddpm_snapshots,
    Config.FIGURES_DIR / "fig5_training_curves_and_ddpm_trajectory.png",
)

print("All 5 high-resolution figures saved to:", Config.FIGURES_DIR)